# GS1 Product Data Transformation Pipeline

Converts a raw supplier product spreadsheet into **GDSN-style XML** with unit → case hierarchies, and writes a **validation report** for any rows that can't be sent.

**Steps**
1. Load the raw supplier file
2. Clean and validate barcodes (GTIN check digits, duplicates)
3. Map sizes to GS1 / UN-ECE unit codes
4. Convert weights to grams
5. Convert dimensions from cm to mm
6. Split product names into brand and functional name; map countries to ISO 3166 codes
7. Run validation rules and build the error report
8. Export clean products to XML

*All data is fictional. GTINs use valid check digits but are not registered products.*

## 1. Load the raw supplier file

`dtype=str` keeps barcodes as text so pandas can't alter them. `keep_default_na=False` stops values like "None" from being read as empty.

In [1]:
import pandas as pd
df = pd.read_excel("Sahil_Foods_Client_Product_Data.xlsx",
                   sheet_name="Client_Data",
                   dtype=str,               # keeps barcodes as text
                   keep_default_na=False)   # stops "None" turning into NaN

### Preview the raw data

In [2]:
df.head(12)

,Barcode,Product Description,Size,Pack Qty,Case Barcode,Unit Dims (cm),Unit Weight,Case Dims (cm),Case Weight,Country of Origin,Allergens
0,8961234500013,Sahil Basmati Rice 5kg,5 kg,4,18961234500010,38x25x12,5.1kg,52x40x26,21 kg,Pakistan,None
1,8961234500020,sahil mango juice,1 Litre,12,18961234500027,25 x 8 x 6,1080 g,33x25x26,13.2kg,PK,
2,8961234500036,SAHIL MASALA CHIPS 50G,50 grams,24,18961234500034,18x12x3,55g,,,586,May contain milk
3,8961234500044,Sahil Chilli Garlic Sauce 300ml,300ML,12,18961234500041,18x6x6,0.42 kg,25x19x19,5.3kg,Pakistan,None
4,8961234500051,Sahil Red Lentils (Masoor Daal) 1kg,1KG,10,18961234500058,25x15x6,1.02kg,,,Pakistan,None
5,8961234500068,Sahil Coconut Cookies 120g,120 g,24,18961234500065,20x6x4,130g,42x26x22,3.4 kg,Pakistan,"Wheat, Milk, Soy"
6,8961234500075,Sahil Mango Juice 250ml,0.25 L,27,18961234500072,11x5x4,0.2 kg,35x22x12,7.1kg,PK,
7,8961234500082,Sahil Pink Salt 800g,800 gm,12,18961234500089,16x8x8,820g,33x25x17,10.1kg,Pakistan,None
8,8961234500020,Sahil Mango Juice 1L - Promo Pack,1 Litre,12,18961234500096,25 x 8 x 6,1080 g,33x25x26,13.2kg,PK,
9,8961234500105,,500 ml,12,18961234500102,20x7x7,540g,29x22x21,6.7kg,Pakistan,None


## 2. Clean and validate barcodes

Strips hidden spaces, checks each GTIN against the GS1 mod-10 check-digit rule, and flags duplicate unit GTINs.

In [3]:
# Remove spaces around barcodes (one row has hidden spaces)
df["Barcode"] = df["Barcode"].str.strip()
df["Case Barcode"] = df["Case Barcode"].str.strip()

def gtin_is_valid(gtin):
    """Return True if the GTIN has a correct GS1 check digit."""
    if not gtin.isdigit() or len(gtin) not in (8, 12, 13, 14):
        return False
    body, check = gtin[:-1], int(gtin[-1])
    total = 0
    for i, digit in enumerate(reversed(body)):
        weight = 3 if i % 2 == 0 else 1   # GS1 rule: weights 3,1,3,1... from the right
        total += int(digit) * weight
    return (10 - total % 10) % 10 == check

df["barcode_valid"] = df["Barcode"].apply(gtin_is_valid)
df["case_barcode_valid"] = df["Case Barcode"].apply(gtin_is_valid)
df["barcode_duplicate"] = df["Barcode"].duplicated(keep=False)

df[["Barcode", "barcode_valid", "case_barcode_valid", "barcode_duplicate"]]

,Barcode,barcode_valid,case_barcode_valid,barcode_duplicate
0,8961234500013,True,True,False
1,8961234500020,True,True,True
2,8961234500036,False,True,False
3,8961234500044,True,True,False
4,8961234500051,True,True,False
5,8961234500068,True,True,False
6,8961234500075,True,True,False
7,8961234500082,True,True,False
8,8961234500020,True,True,True
9,8961234500105,True,True,False


## 3. Map sizes to GS1 / UN-ECE unit codes

Splits free text such as `"300ML"` or `"1 Litre"` into a number and a standard unit code (`MLT`, `LTR`, `GRM`, `KGM`).

In [4]:
import re

UNIT_MAP = {"kg": "KGM", "kilogram": "KGM",
            "g": "GRM", "gm": "GRM", "gram": "GRM", "grams": "GRM",
            "l": "LTR", "litre": "LTR", "liter": "LTR",
            "ml": "MLT", "millilitre": "MLT"}

def parse_size(text):
    """'5 kg' -> (5.0, 'KGM')"""
    match = re.match(r"^\s*([\d.]+)\s*([a-zA-Z]+)\s*$", text)
    if not match:
        return None, None
    number, unit = match.groups()
    return float(number), UNIT_MAP.get(unit.lower())

df[["net_content", "net_content_uom"]] = df["Size"].apply(lambda s: pd.Series(parse_size(s)))
df[["Size", "net_content", "net_content_uom"]]

,Size,net_content,net_content_uom
0,5 kg,5.00,KGM
1,1 Litre,1.00,LTR
2,50 grams,50.00,GRM
3,300ML,300.00,MLT
4,1KG,1.00,KGM
5,120 g,120.00,GRM
6,0.25 L,0.25,LTR
7,800 gm,800.00,GRM
8,1 Litre,1.00,LTR
9,500 ml,500.00,MLT


## 4. Convert weights to grams

Values with no unit (e.g. `"8.6"`) are left empty rather than guessed, so they appear in the validation report.

In [5]:
WEIGHT_TO_GRAMS = {"kg": 1000, "g": 1, "gm": 1, "gram": 1, "grams": 1}

def to_grams(text):
    """'5.1kg' -> 5100, '55g' -> 55. Returns None if the unit is missing."""
    match = re.match(r"^\s*([\d.]+)\s*([a-zA-Z]+)\s*$", text)
    if not match:
        return None
    number, unit = match.groups()
    factor = WEIGHT_TO_GRAMS.get(unit.lower())
    return round(float(number) * factor) if factor else None

df["unit_gross_weight_g"] = df["Unit Weight"].apply(to_grams)
df["case_gross_weight_g"] = df["Case Weight"].apply(to_grams)
df[["Unit Weight", "unit_gross_weight_g", "Case Weight", "case_gross_weight_g"]]

,Unit Weight,unit_gross_weight_g,Case Weight,case_gross_weight_g
0,5.1kg,5100,21 kg,21000.0
1,1080 g,1080,13.2kg,13200.0
2,55g,55,,NaN
3,0.42 kg,420,5.3kg,5300.0
4,1.02kg,1020,,NaN
5,130g,130,3.4 kg,3400.0
6,0.2 kg,200,7.1kg,7100.0
7,820g,820,10.1kg,10100.0
8,1080 g,1080,13.2kg,13200.0
9,540g,540,6.7kg,6700.0


## 5. Convert dimensions from cm to mm

Assumes the client's order is height × width × depth.

In [6]:
def dims_to_mm(text):
    """'38x25x12' (cm) -> (380, 250, 120) in mm, as height, width, depth."""
    parts = [p.strip() for p in text.lower().split("x")]
    if len(parts) != 3 or not all(p.replace(".", "", 1).isdigit() for p in parts):
        return None, None, None
    return tuple(round(float(p) * 10) for p in parts)

df[["unit_h_mm", "unit_w_mm", "unit_d_mm"]] = df["Unit Dims (cm)"].apply(lambda s: pd.Series(dims_to_mm(s)))
df[["case_h_mm", "case_w_mm", "case_d_mm"]] = df["Case Dims (cm)"].apply(lambda s: pd.Series(dims_to_mm(s)))
df[["Unit Dims (cm)", "unit_h_mm", "unit_w_mm", "unit_d_mm", "Case Dims (cm)", "case_h_mm"]]

,Unit Dims (cm),unit_h_mm,unit_w_mm,unit_d_mm,Case Dims (cm),case_h_mm
0,38x25x12,380,250,120,52x40x26,520.0
1,25 x 8 x 6,250,80,60,33x25x26,330.0
2,18x12x3,180,120,30,,NaN
3,18x6x6,180,60,60,25x19x19,250.0
4,25x15x6,250,150,60,,NaN
5,20x6x4,200,60,40,42x26x22,420.0
6,11x5x4,110,50,40,35x22x12,350.0
7,16x8x8,160,80,80,33x25x17,330.0
8,25 x 8 x 6,250,80,60,33x25x26,330.0
9,20x7x7,200,70,70,29x22x21,290.0


## 6. Brand, functional name, and country of origin

Removes the size and any promo suffix from the description, splits brand from functional name, and maps country text to ISO 3166 numeric codes.

In [7]:
SIZE_PATTERN = r"\s*\d+(\.\d+)?\s*(kg|g|gm|ml|l|litre)\b"

def split_name(text):
    """'Sahil Basmati Rice 5kg' -> ('Sahil', 'Basmati Rice')"""
    if not text.strip():
        return None, None
    clean = re.sub(SIZE_PATTERN, "", text, flags=re.IGNORECASE)   # remove "5kg", "300ml"...
    clean = re.sub(r"\s+-\s+.*$", "", clean).strip()               # remove " - Promo Pack"
    words = clean.split()
    return words[0].title(), " ".join(words[1:]).title()

df[["brand_name", "functional_name"]] = df["Product Description"].apply(lambda s: pd.Series(split_name(s)))

COUNTRY_MAP = {"pakistan": "586", "pk": "586", "586": "586",
               "india": "356", "in": "356", "356": "356"}
df["country_of_origin"] = df["Country of Origin"].str.strip().str.lower().map(COUNTRY_MAP)

df[["Product Description", "brand_name", "functional_name", "country_of_origin"]]

,Product Description,brand_name,functional_name,country_of_origin
0,Sahil Basmati Rice 5kg,Sahil,Basmati Rice,586
1,sahil mango juice,Sahil,Mango Juice,586
2,SAHIL MASALA CHIPS 50G,Sahil,Masala Chips,586
3,Sahil Chilli Garlic Sauce 300ml,Sahil,Chilli Garlic Sauce,586
4,Sahil Red Lentils (Masoor Daal) 1kg,Sahil,Red Lentils (Masoor Daal),586
5,Sahil Coconut Cookies 120g,Sahil,Coconut Cookies,586
6,Sahil Mango Juice 250ml,Sahil,Mango Juice,586
7,Sahil Pink Salt 800g,Sahil,Pink Salt,586
8,Sahil Mango Juice 1L - Promo Pack,Sahil,Mango Juice,586
9,,NaN,NaN,586


## 7. Validation rules

Every failed rule becomes one row in the report. Any product with an issue is held back from the XML export.

In [8]:
issues = []

def add_issue(excel_row, gtin, problem):
    issues.append({"excel_row": excel_row, "gtin": gtin, "problem": problem})

df["case_duplicate"] = df["Case Barcode"].duplicated(keep=False)
NET_TO_GRAMS = {"KGM": 1000, "GRM": 1, "LTR": 1000, "MLT": 1}

for i, row in df.iterrows():
    excel_row = i + 2          # +2 because Excel has a header row and starts at 1
    gtin = row["Barcode"]
    if not row["barcode_valid"]:
        add_issue(excel_row, gtin, "Invalid unit GTIN check digit")
    if not row["case_barcode_valid"]:
        add_issue(excel_row, gtin, "Invalid case GTIN check digit")
    if row["barcode_duplicate"]:
        add_issue(excel_row, gtin, "Unit GTIN used on more than one row")
    if row["case_duplicate"]:
        add_issue(excel_row, gtin, "Case GTIN used on more than one row")
    if pd.isna(row["brand_name"]):
        add_issue(excel_row, gtin, "Missing product description")
    if pd.isna(row["case_gross_weight_g"]):
        add_issue(excel_row, gtin, "Case weight missing or has no unit")
    if pd.isna(row["case_h_mm"]):
        add_issue(excel_row, gtin, "Case dimensions missing")
    net_g = row["net_content"] * NET_TO_GRAMS[row["net_content_uom"]]
    if row["unit_gross_weight_g"] < net_g:
        add_issue(excel_row, gtin, f"Gross weight ({row['unit_gross_weight_g']} g) is less than net content ({row['Size']})")

issues_df = pd.DataFrame(issues)
issues_df

,excel_row,gtin,problem
0,2,8961234500013,Case GTIN used on more than one row
1,3,8961234500020,Unit GTIN used on more than one row
2,4,8961234500036,Invalid unit GTIN check digit
3,4,8961234500036,Case weight missing or has no unit
4,4,8961234500036,Case dimensions missing
5,6,8961234500051,Case weight missing or has no unit
6,6,8961234500051,Case dimensions missing
7,8,8961234500075,Gross weight (200 g) is less than net content ...
8,10,8961234500020,Unit GTIN used on more than one row
9,11,8961234500105,Missing product description


## 8. Export clean products to XML

Each clean product produces two trade items: the consumer unit (`BASE_UNIT_OR_EACH`) and its case, linked through `childTradeItem`. The validation report is saved as Excel so barcodes aren't shown in scientific notation.

In [9]:
import xml.etree.ElementTree as ET

SUPPLIER_GLN = "8961234000001"
TARGET_MARKET = "826"   # United Kingdom

def add(parent, tag, value, unit=None):
    """Add <tag>value</tag> to parent; optional unit becomes measurementUnitCode."""
    element = ET.SubElement(parent, tag)
    element.text = str(value)
    if unit:
        element.set("measurementUnitCode", unit)
    return element

def build_trade_item(parent, row, level):
    item = ET.SubElement(parent, "tradeItem")
    is_unit = level == "unit"
    add(item, "gtin", row["Barcode"] if is_unit else row["Case Barcode"])
    add(item, "tradeItemUnitDescriptorCode", "BASE_UNIT_OR_EACH" if is_unit else "CASE")
    add(item, "isTradeItemAConsumerUnit", "true" if is_unit else "false")
    provider = ET.SubElement(item, "informationProviderOfTradeItem")
    add(provider, "gln", SUPPLIER_GLN)
    market = ET.SubElement(item, "targetMarket")
    add(market, "targetMarketCountryCode", TARGET_MARKET)
    add(item, "brandName", row["brand_name"])
    add(item, "functionalName", row["functional_name"])
    add(item, "countryOfOrigin", row["country_of_origin"])
    if is_unit:
        add(item, "netContent", f"{row['net_content']:g}", row["net_content_uom"])
        h, w, d, weight = row["unit_h_mm"], row["unit_w_mm"], row["unit_d_mm"], row["unit_gross_weight_g"]
    else:
        child_info = ET.SubElement(item, "nextLowerLevelTradeItemInformation")
        child = ET.SubElement(child_info, "childTradeItem")
        add(child, "gtin", row["Barcode"])
        add(child, "quantityOfNextLowerLevelTradeItem", row["Pack Qty"])
        h, w, d, weight = row["case_h_mm"], row["case_w_mm"], row["case_d_mm"], row["case_gross_weight_g"]
    add(item, "height", int(h), "MMT")
    add(item, "width", int(w), "MMT")
    add(item, "depth", int(d), "MMT")
    add(item, "grossWeight", int(weight), "GRM")

# Only export rows with no issues; everything else waits for the client to fix it
rows_with_issues = set(issues_df["excel_row"]) if len(issues_df) else set()
catalogue = ET.Element("catalogue")
exported = 0
for i, row in df.iterrows():
    if i + 2 in rows_with_issues:
        continue
    build_trade_item(catalogue, row, "unit")
    build_trade_item(catalogue, row, "case")
    exported += 1

ET.indent(catalogue)
ET.ElementTree(catalogue).write("sahil_foods_gdsn.xml", encoding="utf-8", xml_declaration=True)
issues_df.to_excel("validation_report.xlsx", index=False)

print(f"Exported {exported} products ({exported * 2} trade items) to sahil_foods_gdsn.xml")
print(f"Held back {len(rows_with_issues)} rows; see validation_report.xlsx")

Exported 3 products (6 trade items) to sahil_foods_gdsn.xml
Held back 9 rows; see validation_report.xlsx


## 9. Round-trip check

Read the XML back and confirm the output matches the source.

In [10]:
check = pd.read_xml("sahil_foods_gdsn.xml", xpath="//tradeItem", dtype={"gtin": str})
check[["gtin", "tradeItemUnitDescriptorCode", "functionalName",
       "netContent", "height", "width", "depth", "grossWeight"]]

,gtin,tradeItemUnitDescriptorCode,functionalName,netContent,height,width,depth,grossWeight
0,8961234500044,BASE_UNIT_OR_EACH,Chilli Garlic Sauce,300.0,180,60,60,420
1,18961234500041,CASE,Chilli Garlic Sauce,NaN,250,190,190,5300
2,8961234500068,BASE_UNIT_OR_EACH,Coconut Cookies,120.0,200,60,40,130
3,18961234500065,CASE,Coconut Cookies,NaN,420,260,220,3400
4,8961234500082,BASE_UNIT_OR_EACH,Pink Salt,800.0,160,80,80,820
5,18961234500089,CASE,Pink Salt,NaN,330,250,170,10100
